# RAG Assignment — Employment Termination Disputes in Kenya's ELRC

## README

- **Topic Name:** Employment termination and wrongful dismissal disputes in Kenya's Employment and Labour Relations Court (ELRC)
- **Source & Licence Type:** Judgments and rulings downloaded from Kenya Law (kenyalaw.org / new.kenyalaw.org), the National Council for Law Reporting. Per Kenya Law's website footer: *"Except for some material which is expressly stated to be under a specified Creative Commons license, the contents of this website are in the public domain and free from any copyright restrictions."* This means these judgments can be freely reused for this academic project.
- **Embedding Model Used:** `sentence-transformers/all-MiniLM-L6-v2` (open-source, runs locally, 384-dimensional vectors)
- **LLM Used:** Google Gemini (`gemini-2.0-flash`) via the Gemini API
- **Vector Database Used:** Chroma (open-source, persisted locally to disk)
- **Execution Instructions:**
    1. Place your Kenya Law PDF files inside a folder called `data/raw/` next to this notebook (51 files already provided).
    2. Create a `.env` file in the same folder as this notebook containing: `GEMINI_API_KEY=your_key_here`
    3. Run the cells from top to bottom (Kernel → Restart & Run All).
    4. The Chroma database will be created inside `chroma_db/` the first time you run it, and reloaded from disk on later runs (no re-embedding needed).


## Task 0: Topic & Source Selection

**Topic (one sentence):** Employment termination and wrongful dismissal disputes decided by Kenya's Employment and Labour Relations Court (ELRC).

**Who would use this system and why:** Law students, paralegals, or HR professionals in Kenya who want to quickly check how the ELRC has ruled on employment termination issues (e.g. notice periods, summary dismissal, redundancy) without manually reading through dozens of long judgment PDFs.

**3 example questions the system should answer:**
1. What did the court decide about summary dismissal without a hearing in these cases?
2. What compensation did the court award in cases involving redundancy without notice?
3. What factors does the court consider when deciding if a dismissal was procedurally unfair?

**1 question it should NOT be able to answer (out of scope):**
- "What is the maximum penalty for tax evasion in Kenya?" — this dataset only contains ELRC employment judgments, not tax law, so the system should say it cannot answer this.

**Source chosen:** Kenya Law (kenyalaw.org) — Case law (ELRC judgments and rulings).


## Task 1: Document Collection & Logging

We already have 51 PDF judgments/rulings downloaded from Kenya Law and placed in `data/raw/`. The cell below:
1. Lists all the files and confirms the count.
2. Extracts basic stats (word/page counts, file type breakdown) using `pypdf`.
3. Builds a Collection Log Table with one row per file.
4. Prints the first 500 characters of 3 documents to check the text is readable.


In [1]:
# --- Imports we'll need for Task 1 ---
import os
import glob
from pathlib import Path
import pandas as pd
from pypdf import PdfReader

# Folder where the raw PDF documents live
RAW_DIR = Path("data/raw")

print("Looking for PDF files in:", RAW_DIR.resolve())


Looking for PDF files in: /home/claude/rag_project/data/raw


In [2]:
# --- Step 1: List all raw files and confirm the count ---
raw_files = sorted(RAW_DIR.glob("*.pdf"))

print(f"Found {len(raw_files)} PDF files.")
for f in raw_files[:5]:
    print(" -", f.name)
print("...")


Found 51 PDF files.
 - Amalgamated Union of Kenya Metal Workers v Kenya Coach Industries 2020KEELRC148(KLR).pdf
 - Andika v Smart Printers Limited (Cause 460of2018) 2023KEELRC2425(KLR) (6October2023) (Judgment).pdf
 - Angila v Makini School Limited (Cause 586of2019) 2022KEELRC4057(KLR) (30June2022) (Judgment).pdf
 - Antonina Chebet v Pure Circle Kenya Limited 2021KEELRC869(KLR).pdf
 - Bakery Confectionery Food Manufacturing  Allied Workers Union (K) v Bigcold Kenya Limited (Cause E180of2024) 2024KEELRC1809(KLR) (8July2024) (Ruling).pdf
...


In [3]:
# --- Step 2: Go through every PDF and collect stats + build the log table ---
# For each file we record: filename, source_url, date_accessed, type, size, licence_or_terms

# NOTE: Kenya Law doesn't expose one fixed URL per downloaded PDF once it's on your
# computer, so we record the general collection page as the source_url for every row.
# If you want exact per-case links later, you can look each one up on new.kenyalaw.org
# and update the "source_url" column.
GENERAL_SOURCE_URL = "https://new.kenyalaw.org/judgments/KEELRC/"
LICENCE_TEXT = "Public domain / free from copyright restrictions (Kenya Law website terms)"
DATE_ACCESSED = "2026-09-21"  # the date these files were downloaded

log_rows = []
total_words = 0
total_pages = 0
failed_files = []

for f in raw_files:
    try:
        reader = PdfReader(str(f))
        num_pages = len(reader.pages)

        # Pull text out of every page so we can count words
        full_text = ""
        for page in reader.pages:
            page_text = page.extract_text() or ""
            full_text += page_text + " "

        num_words = len(full_text.split())
        total_words += num_words
        total_pages += num_pages

        log_rows.append({
            "filename": f.name,
            "source_url": GENERAL_SOURCE_URL,
            "date_accessed": DATE_ACCESSED,
            "type": "PDF",
            "size_kb": round(f.stat().st_size / 1024, 1),
            "licence_or_terms": LICENCE_TEXT,
            "num_pages": num_pages,
            "num_words": num_words,
        })
    except Exception as e:
        # If a PDF is corrupted or can't be read, log it instead of crashing
        failed_files.append((f.name, str(e)))

# Turn the log into a DataFrame (an easy-to-read table)
collection_log = pd.DataFrame(log_rows)

print(f"Successfully processed: {len(collection_log)} documents")
print(f"Failed to process: {len(failed_files)} documents")
if failed_files:
    print("Failed files:", failed_files)


Successfully processed: 51 documents
Failed to process: 0 documents


In [4]:
# --- Step 3: Print overall dataset statistics ---
file_type_counts = collection_log["type"].value_counts()

print("Dataset statistics")
print("-------------------")
print(f"Total documents : {len(collection_log)}")
print(f"Total pages      : {total_pages}")
print(f"Total words      : {total_words}")
print()
print("File type breakdown:")
print(file_type_counts)


Dataset statistics
-------------------
Total documents : 51
Total pages      : 332
Total words      : 164808

File type breakdown:
type
PDF    51
Name: count, dtype: int64


In [5]:
# --- Step 4: Display the Collection Log Table ---
# Showing the columns the assignment asks for, in order
display_cols = ["filename", "source_url", "date_accessed", "type", "size_kb", "licence_or_terms"]
collection_log_display = collection_log[display_cols].rename(columns={"size_kb": "size"})
collection_log_display


,filename,source_url,date_accessed,type,size,licence_or_terms
0,Amalgamated Union of Kenya Metal Workers v Ken...,https://new.kenyalaw.org/judgments/KEELRC/,2026-09-21,PDF,157.7,Public domain / free from copyright restrictio...
1,Andika v Smart Printers Limited (Cause 460of20...,https://new.kenyalaw.org/judgments/KEELRC/,2026-09-21,PDF,228.6,Public domain / free from copyright restrictio...
2,Angila v Makini School Limited (Cause 586of201...,https://new.kenyalaw.org/judgments/KEELRC/,2026-09-21,PDF,216.0,Public domain / free from copyright restrictio...
3,Antonina Chebet v Pure Circle Kenya Limited 20...,https://new.kenyalaw.org/judgments/KEELRC/,2026-09-21,PDF,148.9,Public domain / free from copyright restrictio...
4,Bakery Confectionery Food Manufacturing Allie...,https://new.kenyalaw.org/judgments/KEELRC/,2026-09-21,PDF,212.6,Public domain / free from copyright restrictio...
5,Benson Omuyonga v Laxmanbhai Construction Ltd ...,https://new.kenyalaw.org/judgments/KEELRC/,2026-09-21,PDF,149.1,Public domain / free from copyright restrictio...
6,George Buoro v Kenindia Assurance Company 2020...,https://new.kenyalaw.org/judgments/KEELRC/,2026-09-21,PDF,159.0,Public domain / free from copyright restrictio...
7,Gituma v PZ Cussons PLC (Petition E178of2021) ...,https://new.kenyalaw.org/judgments/KEELRC/,2026-09-21,PDF,243.2,Public domain / free from copyright restrictio...
8,Godfrey Andabwa Ashiono v Coconut (K) Limited ...,https://new.kenyalaw.org/judgments/KEELRC/,2026-09-21,PDF,160.9,Public domain / free from copyright restrictio...
9,Harun v Watu Credit Limited (Employment and La...,https://new.kenyalaw.org/judgments/KEELRC/,2026-09-21,PDF,211.4,Public domain / free from copyright restrictio...


In [6]:
# --- Step 5: Save the log table to disk so we have a permanent record ---
os.makedirs("data", exist_ok=True)
collection_log.to_csv("data/collection_log.csv", index=False)
print("Saved collection log to data/collection_log.csv")


Saved collection log to data/collection_log.csv


In [7]:
# --- Step 6: Print the first 500 characters of 3 different documents ---
# This lets us eyeball whether the PDF text extraction actually worked (readable text,
# not garbled symbols).
sample_files = raw_files[:3]

for f in sample_files:
    reader = PdfReader(str(f))
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "")
        if len(text) >= 500:
            break

    print("=" * 80)
    print("FILE:", f.name)
    print("-" * 80)
    print(text[:500])
    print()


FILE: Amalgamated Union of Kenya Metal Workers v Kenya Coach Industries 2020KEELRC148(KLR).pdf
--------------------------------------------------------------------------------
REPUBLIC OF KENYA
IN THE EMPLOYMENT AND LABOUR RELATIONS COURT AT NAIROBI
CAUSE NO. E450 OF 2020
(Before Hon. Lady Justice Maureen Onyango)
AMALGAMATED UNION OF 
KENYA METAL WORKERS...................................CLAIMANT
VERSUS
KENYA COACH INDUSTRIES..........................RESPONDENT
RULING
The claimant herein is a trade union registered under the Labour Relations Act while the Respondent operates within the Sector under which
the Claimant union is empowered by its Constitution to recruit 

FILE: Andika v Smart Printers Limited (Cause 460of2018) 2023KEELRC2425(KLR) (6October2023) (Judgment).pdf
--------------------------------------------------------------------------------
Andika v Smart Printers Limited (Cause 460 of 2018)
[2023] KEELRC 2425 (KLR) (6 October 2023) (Judgment)
Neutral citation: [2023] KEELR

## Task 2: Cleaning (Optional but Recommended)

Before deciding whether to clean the text, we inspect 5 documents in more detail — checking for
things like broken encoding, repeated headers/footers on every page, or excessive whitespace.
The cell below runs those checks automatically instead of just eyeballing it.

In [8]:
# --- Inspect 5 documents for common "needs cleaning" problems ---
import re

sample_for_inspection = raw_files[:5]

for f in sample_for_inspection:
    reader = PdfReader(str(f))
    full_text = ""
    for page in reader.pages:
        full_text += (page.extract_text() or "") + "\n"

    # Check 1: are there odd/garbled characters (a sign of broken PDF extraction)?
    non_ascii_count = len(re.findall(r"[^\x00-\x7F]", full_text))
    non_ascii_ratio = non_ascii_count / max(len(full_text), 1)

    # Check 2: how much whitespace/blank-line clutter is there?
    excess_blank_lines = len(re.findall(r"\n{3,}", full_text))

    # Check 3: does the same line repeat many times (a sign of a running header/footer)?
    lines = [l.strip() for l in full_text.split("\n") if l.strip()]
    from collections import Counter
    line_counts = Counter(lines)
    most_repeated_line, repeat_count = line_counts.most_common(1)[0] if line_counts else ("", 0)

    print("=" * 80)
    print("FILE:", f.name)
    print(f"  Non-ASCII character ratio : {non_ascii_ratio:.4%}")
    print(f"  Excess blank-line blocks   : {excess_blank_lines}")
    print(f"  Most repeated line         : \"{most_repeated_line[:60]}\" (appears {repeat_count}x)")


FILE: Amalgamated Union of Kenya Metal Workers v Kenya Coach Industries 2020KEELRC148(KLR).pdf
  Non-ASCII character ratio : 0.4381%
  Excess blank-line blocks   : 0
  Most repeated line         : "MAUREEN ONYANGO" (appears 2x)


FILE: Andika v Smart Printers Limited (Cause 460of2018) 2023KEELRC2425(KLR) (6October2023) (Judgment).pdf
  Non-ASCII character ratio : 0.5792%
  Excess blank-line blocks   : 0
  Most repeated line         : "Andika v Smart Printers Limited (Cause 460 of 2018)" (appears 1x)


FILE: Angila v Makini School Limited (Cause 586of2019) 2022KEELRC4057(KLR) (30June2022) (Judgment).pdf
  Non-ASCII character ratio : 0.8503%
  Excess blank-line blocks   : 0
  Most repeated line         : "Angila v Makini School Limited (Cause 586 of 2019)" (appears 1x)
FILE: Antonina Chebet v Pure Circle Kenya Limited 2021KEELRC869(KLR).pdf
  Non-ASCII character ratio : 0.1686%
  Excess blank-line blocks   : 0
  Most repeated line         : "REPUBLIC OF KENYA" (appears 1x)


FILE: Bakery Confectionery Food Manufacturing  Allied Workers Union (K) v Bigcold Kenya Limited (Cause E180of2024) 2024KEELRC1809(KLR) (8July2024) (Ruling).pdf
  Non-ASCII character ratio : 0.7697%
  Excess blank-line blocks   : 0
  Most repeated line         : "JUDGE" (appears 2x)


### Cleaning decision: SKIPPED

Looking at the checks above across 5 sample documents, the non-ASCII character ratio is close
to 0% (meaning the text extracted correctly, no garbled symbols), there are very few excess
blank-line blocks, and no single line repeats often enough to be a disruptive running
header/footer. In other words: `pypdf` already extracts clean, readable text straight from
these Kenya Law PDFs. Because of this, we chose **not** to run an additional cleaning step —
running normalization/deduplication on text that's already clean would only add complexity
without improving chunk or embedding quality. We do still trim leading/trailing whitespace
per chunk in Task 3, which is a normal part of chunking rather than a separate cleaning
stage.

## Task 3: Extraction, Chunking & Metadata

Now we split every judgment into smaller, retrievable pieces ("chunks") that we can later
embed and search over. The steps here are:
1. Extract text from every PDF **page by page** (not just as one giant blob), so we can
   later tell which page a chunk came from.
2. Try **two different chunking strategies** and compare them with real numbers.
3. Pick one strategy and explain why, using evidence from our own data.
4. Attach metadata (filename, chunk ID, page number) to every chunk.
5. Print 5 example chunks so we can see exactly what we built.

In [9]:
# --- Step 1: Extract text page-by-page, keeping track of which characters belong to which page ---
# We need this so that later, when we cut the text into chunks, we can say
# "this chunk came from page 3" instead of just "this chunk came from the document".

document_texts = []  # one entry per document

for f in raw_files:
    reader = PdfReader(str(f))
    full_text = ""
    page_boundaries = []  # list of (start_char, end_char, page_number) for this document

    for page_num, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""
        start = len(full_text)
        full_text += page_text + "\n"
        end = len(full_text)
        page_boundaries.append((start, end, page_num))

    document_texts.append({
        "filename": f.name,
        "full_text": full_text,
        "page_boundaries": page_boundaries,
    })

# Log any documents where extraction produced nothing (empty or failed extraction)
empty_extractions = [d["filename"] for d in document_texts if len(d["full_text"].strip()) == 0]

print(f"Documents processed        : {len(document_texts)}")
print(f"Documents with EMPTY text  : {len(empty_extractions)}")
if empty_extractions:
    print("  ->", empty_extractions)


Documents processed        : 51
Documents with EMPTY text  : 0


In [10]:
# --- Helper: given a character offset, find which page it belongs to ---
def find_page_number(offset, page_boundaries):
    for start, end, page_num in page_boundaries:
        if start <= offset < end:
            return page_num
    # If we somehow fall past the end, just say it's the last page
    return page_boundaries[-1][2] if page_boundaries else 1


### Strategy A: Character-based chunking (fixed size, with overlap)

This strategy just cuts the text into fixed-length pieces (e.g. every 1000 characters),
sliding forward with some overlap so we don't lose context at the boundary between chunks.

In [11]:
# --- Strategy A: Character-based chunking ---
def chunk_by_characters(text, chunk_size=1000, overlap=150):
    """Cuts `text` into pieces of `chunk_size` characters, each one overlapping
    the previous one by `overlap` characters. Returns a list of (start_offset, chunk_text)."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        piece = text[start:end].strip()
        if piece:  # skip empty pieces (can happen at the very end)
            chunks.append((start, piece))
        start += chunk_size - overlap  # slide forward, but overlap with the previous chunk
    return chunks


### Strategy B: Sentence-based chunking

This strategy first splits the text into individual sentences, then groups consecutive
sentences together until adding one more would go over our target size.

In [12]:
# --- Strategy B: Sentence-based chunking ---
import re

def split_into_sentences_with_spans(text):
    """A simple sentence splitter: treats a run of characters ending in . ! or ? as one
    sentence. Returns a list of (sentence_text, start_offset, end_offset)."""
    spans = []
    for match in re.finditer(r"[^.!?]+[.!?]*", text):
        sentence = match.group().strip()
        if sentence:
            spans.append((sentence, match.start(), match.end()))
    return spans

def chunk_by_sentences(text, target_size=1000):
    """Groups sentences together until the group would exceed `target_size` characters,
    then starts a new group. Returns a list of (start_offset, chunk_text)."""
    sentences = split_into_sentences_with_spans(text)
    chunks = []
    current_text = ""
    current_start = None

    for sentence, start, end in sentences:
        if current_start is None:
            current_start = start

        if not current_text or len(current_text) + len(sentence) <= target_size:
            current_text += (" " if current_text else "") + sentence
        else:
            chunks.append((current_start, current_text.strip()))
            current_text = sentence
            current_start = start

    if current_text:
        chunks.append((current_start, current_text.strip()))

    return chunks


### Comparing the two strategies

We run both strategies over the full set of 51 documents and compare the resulting chunk
counts and sizes.

In [13]:
# --- Run both strategies across every document and collect size statistics ---
CHUNK_SIZE = 1000     # characters
CHUNK_OVERLAP = 150   # characters (used only by Strategy A)

char_chunk_lengths = []
sentence_chunk_lengths = []

for doc in document_texts:
    text = doc["full_text"]

    char_chunks = chunk_by_characters(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    char_chunk_lengths.extend(len(c[1]) for c in char_chunks)

    sentence_chunks = chunk_by_sentences(text, target_size=CHUNK_SIZE)
    sentence_chunk_lengths.extend(len(c[1]) for c in sentence_chunks)

def summarize(lengths, name):
    lengths_series = pd.Series(lengths)
    print(f"{name}")
    print(f"  Total chunks : {len(lengths_series)}")
    print(f"  Mean length  : {lengths_series.mean():.0f} characters")
    print(f"  Min / Max    : {lengths_series.min()} / {lengths_series.max()} characters")
    print(f"  Std deviation: {lengths_series.std():.0f} characters")
    print()

summarize(char_chunk_lengths, "Strategy A: Character-based chunking")
summarize(sentence_chunk_lengths, "Strategy B: Sentence-based chunking")


Strategy A: Character-based chunking
  Total chunks : 1230
  Mean length  : 976 characters
  Min / Max    : 5 / 1000 characters
  Std deviation: 124 characters

Strategy B: Sentence-based chunking
  Total chunks : 1146
  Mean length  : 893 characters
  Min / Max    : 35 / 1573 characters
  Std deviation: 141 characters



### Chosen strategy: Character-based chunking (Strategy A)

Running both strategies over all 51 documents gave us real numbers to compare (see the output
above): Strategy A (character-based) produced chunks capped at exactly 1000 characters, with a
standard deviation of 124. Strategy B (sentence-based) produced chunks with a **higher maximum
(1573 characters, well above our 1000-character target)** and a **higher standard deviation
(141)** — this happens because legal judgments often contain very long sentences packed with
semicolons and sub-clauses, so a single "sentence" can end up much longer than intended.

That overflow matters here: our embedding model, `all-MiniLM-L6-v2`, has a maximum input
length of 256 tokens (roughly 1000-1100 characters). A 1573-character chunk risks being
silently truncated by the embedding model, meaning we'd lose information without knowing it.
Strategy A's hard 1000-character cap guarantees every chunk fits safely inside that limit,
while the 150-character overlap still helps avoid cutting off meaning right at a chunk
boundary. This is why we're moving forward with Strategy A for the rest of the pipeline.

### Attaching metadata to every chunk

Now we build the final list of chunks using Strategy A, and attach the metadata the
assignment asks for: `filename`, `chunk_id`, and `page_number` (found using the page
boundaries we tracked in Step 1).

In [14]:
# --- Build the final chunk list with metadata attached ---
final_chunks = []
chunk_counter = 0

for doc in document_texts:
    text = doc["full_text"]
    doc_chunks = chunk_by_characters(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

    for start_offset, chunk_text in doc_chunks:
        chunk_counter += 1
        page_num = find_page_number(start_offset, doc["page_boundaries"])

        final_chunks.append({
            "chunk_id": f"chunk_{chunk_counter:05d}",
            "filename": doc["filename"],
            "page_number": page_num,
            "position": start_offset,   # character offset within the document (useful for debugging)
            "text": chunk_text,
        })

print(f"Total chunks created across all 51 documents: {len(final_chunks)}")


Total chunks created across all 51 documents: 1230


In [15]:
# --- Print 5 example chunks with their attached metadata ---
for chunk in final_chunks[:5]:
    print("=" * 80)
    print(f"chunk_id     : {chunk['chunk_id']}")
    print(f"filename     : {chunk['filename']}")
    print(f"page_number  : {chunk['page_number']}")
    print(f"position     : {chunk['position']}")
    print(f"text preview : {chunk['text'][:200]}...")
    print()


chunk_id     : chunk_00001
filename     : Amalgamated Union of Kenya Metal Workers v Kenya Coach Industries 2020KEELRC148(KLR).pdf
page_number  : 1
position     : 0
text preview : REPUBLIC OF KENYA
IN THE EMPLOYMENT AND LABOUR RELATIONS COURT AT NAIROBI
CAUSE NO. E450 OF 2020
(Before Hon. Lady Justice Maureen Onyango)
AMALGAMATED UNION OF 
KENYA METAL WORKERS......................

chunk_id     : chunk_00002
filename     : Amalgamated Union of Kenya Metal Workers v Kenya Coach Industries 2020KEELRC148(KLR).pdf
page_number  : 1
position     : 850
text preview : would be
reduced by 20% for a period of 2 months as they observe the situation brought about by the COVID-19 pandemic and that in the event of
termination, the terminal dues will be computed in accord...

chunk_id     : chunk_00003
filename     : Amalgamated Union of Kenya Metal Workers v Kenya Coach Industries 2020KEELRC148(KLR).pdf
page_number  : 1
position     : 1700
text preview : or intimidation of the Claimant members durin